### Fact Tables

In [0]:
store_silver = spark.table("02_silver_catalog.silver.store")
order_silver = spark.table("02_silver_catalog.silver.order")
estimate_silver = spark.table("02_silver_catalog.silver.estimate")
invoice_silver = spark.table("02_silver_catalog.silver.invoice")
ns_budget_silver = spark.table("02_silver_catalog.silver.ns_budget")


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
gold_catalog = "03_gold_catalog.dim_fact_tables."

In [0]:
dim_store = (
    store_silver
    .select(
        "store_id",
        "store_name",
        "city",
        "state",
        "manager_id",
        "manager_name",
        "opened_year",
        "store_type"
    )
)

print(f"dim_store: {dim_store.count()} rows, {len(dim_store.columns)} columns")
display(dim_store.limit(5))

dim_store.write.format("delta").mode("overwrite").saveAsTable(gold_catalog+"dim_store")

In [0]:
dim_technician = (
    order_silver
    .filter(F.col("technician_id").isNotNull())
    .select("technician_id", "technician_name")
    .distinct()
    .dropDuplicates(["technician_id"])
    .orderBy("technician_id")
)

print(f"dim_technician: {dim_technician.count()} rows, {len(dim_technician.columns)} columns")
display(dim_technician.limit(5))

dim_technician.write.format("delta").mode("overwrite").saveAsTable(gold_catalog+"dim_technician")

In [0]:
dim_estimator = (
    estimate_silver
    .filter(F.col("estimator_id").isNotNull())
    .select("estimator_id", "estimator_name")
    .distinct()
    .dropDuplicates(["estimator_id"])
)

print(f"dim_estimator: {dim_estimator.count()} rows, {len(dim_estimator.columns)} columns")

dim_estimator.write.format("delta").mode("overwrite").saveAsTable(gold_catalog+"dim_estimator")

In [0]:
# Remove duplicates by keeping first make/model per vehicle_no
dim_vehicle = (
    order_silver
    .select("vehicle_no", "vehicle_make", "vehicle_model")
    .dropDuplicates(["vehicle_no"])  # Keep first occurrence of each vehicle_no
)

print(f"dim_vehicle: {dim_vehicle.count()} rows (deduplicated)")

dim_vehicle.write.format("delta").mode("overwrite").saveAsTable(gold_catalog+"dim_vehicle")

In [0]:
min_date = (
    order_silver
        .select(F.min(F.to_date("vehicle_in_datetime")).alias("min_date"))
        .first()[0]
)

max_date = (
    invoice_silver
        .select(F.max("invoice_date").alias("max_date"))
        .first()[0]
)

print("MIN DATE:", min_date)
print("MAX DATE:", max_date)

if min_date is None or max_date is None:
    raise ValueError("Cannot build dim_date because min_date or max_date is NULL.")

date_df = (
    spark.range(0, (max_date - min_date).days + 1)
         .withColumn("id_int", F.col("id").cast("int"))
         .withColumn("date", F.expr(f"date_add('{min_date}', id_int)"))
         .drop("id", "id_int")
)

dim_date = (
    date_df
        .withColumn("date_key", F.date_format("date", "yyyyMMdd"))
        .withColumn("year", F.year("date"))
        .withColumn("month", F.month("date"))
        .withColumn("day", F.dayofmonth("date"))
        .withColumn("quarter", F.quarter("date"))
        .withColumn("week_of_year", F.weekofyear("date"))
        .withColumn("day_of_week", F.date_format("date", "E"))
        .withColumn("is_weekend", F.col("day_of_week").isin("Sat", "Sun"))
)

print(f"dim_date: {dim_date.count()} rows")

dim_date.write.format("delta").mode("overwrite").saveAsTable(gold_catalog+"dim_date")

In [0]:
# Dimension table using composite natural key (store_id, budget_year, budget_month)
dim_budget = (
    ns_budget_silver
    .select(
        F.col("ns_store_id").alias("store_id"),
        "budget_year",
        "budget_month",
        "budget_date",
        "budget_amount",
        "approved_by"
    )
)

print(f"dim_budget: {dim_budget.count()} rows, {len(dim_budget.columns)} columns")
display(dim_budget.limit(5))

dim_budget.write.format("delta").mode("overwrite").saveAsTable(gold_catalog+"dim_budget")